In [ ]:
import polars as pl

: 

In [ ]:
basics = pl.read_csv("../datasets/title.basics0.tsv.gz", separator="\t", null_values=r"\N", quote_char=None)
ratings = pl.read_csv("../datasets/title.ratings.tsv.gz", separator="\t", null_values=r"\N")
principals = pl.read_csv("../datasets/title.principals0.tsv.gz", separator="\t", null_values=r"\N")

In [ ]:
# 1.
(
    basics
    .filter(pl.col("titleType") == "movie")
    .join(ratings, on="tconst")
    .sort(["averageRating", "numVotes"], descending=[True, True])
    .select(["primaryTitle", "averageRating", "numVotes"])
    .limit(5)
)

primaryTitle,averageRating,numVotes
str,f64,i64
"""Kaveri""",10.0,1023
"""Kurukku""",10.0,451
"""Jedal Dar Omghe 30 Metri""",10.0,142
"""Sargashte""",10.0,134
"""Gorgeous Rascal""",10.0,115


In [ ]:
# 2.

(
    basics
    .filter(pl.col("titleType") == "movie")
    .join(ratings, on="tconst")
    .filter(pl.col("averageRating") > 8)
    .with_columns(pl.col("genres").str.split(","))
    .explode("genres")
    .filter(pl.col("genres").is_not_null())
    .group_by("genres")
    .len()
    .sort("len", descending=True)
    .limit(1)
)

genres,len
str,u32
"""Documentary""",10564


In [ ]:
# 3.

high_movies = (
    basics
    .filter(pl.col("titleType") == "movie")
    .join(ratings, on="tconst")
    .filter(pl.col("averageRating") > 7.5)
    .select("tconst")
)

(
    principals
    .filter(pl.col("category") == "actor")
    .join(high_movies, on="tconst")
    .group_by("nconst")
    .len()
    .sort("len", descending=True)
    .limit(3)
)

nconst,len
str,u32
"""nm0004660""",231
"""nm0595934""",155
"""nm3183374""",124
